# Scratch Training
### From Stage to System: Bias Propagation in Clinical AI Pipelines
**Hilina Fissha Woreta**

**Luwam Major Kefali**

This notebook extends the Stage 1 sub-analysis by training XGBoost from scratch on the 3,986-patient cohort. Both models are trained on the same data and with the same algorithm; the only difference is that Model B includes the 9 discharge note features as additional inputs. 

## Setup

In [ ]:
import numpy as np
import pandas as pd
import xgboost as xgb
import optuna
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.isotonic import IsotonicRegression

optuna.logging.set_verbosity(optuna.logging.WARNING)

random_seed = 42

preprocessing_dir = "/kaggle/input/notebooks/hilinafissha16/preprocessing"
stage1_path       = "/kaggle/input/datasets/hilinafissha16/mimic-notes/stage1_extracted.parquet"

print("setup complete")

## Loading the data

Stacking all three splits and filtering to the patients we have discharge note features for.

In [ ]:
splits = []
for split_name in ["train", "val", "test"]:
    df = pd.read_parquet(f"{preprocessing_dir}/splits/{split_name}.parquet")
    df["split"] = split_name
    splits.append(df)

all_data = pd.concat(splits, ignore_index=True)

stage1 = pd.read_parquet(stage1_path)

cohort = all_data[all_data["hadm_id"].isin(stage1["hadm_id"])].copy()
cohort = cohort.reset_index(drop=True)
cohort = cohort.merge(stage1, on="hadm_id", how="left")

print("cohort size:", len(cohort))
print("race distribution:")
print((cohort["race_clean"].value_counts(normalize=True) * 100).round(1))
print("readmission rate:", round(cohort["readmitted_30d"].mean() * 100, 1), "%")

## LLM feature variance check

Confirming that the discharge note features have real variance before training. Near-constant features would be uninformative and could indicate extraction problems.

In [ ]:
llm_check_cols = [
    "comorbidity_burden_score",
    "discharge_risk_indicator_count",
    "sdoh_housing_instability",
    "sdoh_food_insecurity",
    "sdoh_substance_use",
    "sdoh_limited_social_support",
    "sdoh_unemployment",
    "sdoh_transportation_barrier",
]

print("LLM feature means:")
print(cohort[llm_check_cols].mean().round(3))

print("\nvalue counts for binary SDOH features:")
for col in [c for c in llm_check_cols if "sdoh_" in c]:
    print(f"  {col}: {cohort[col].value_counts().to_dict()}")

## Discharge note columns: what we use and what we drop

The Stage 1 LLM extraction produces 12 columns per patient. Not all of them are usable as model features.

**Used as features (9 columns):**
- `comorbidity_burden_score` — a 0–10 numeric score summarising how many and how severe the patient's chronic conditions are, as described in the discharge note
- `psychiatric_complexity` — a string label (low / medium / high) indicating the complexity of the patient's psychiatric history; re-encoded as a numeric scale (0/1/2) before training
- `discharge_risk_indicator_count` — a count of how many distinct readmission risk factors were mentioned in the note
- `sdoh_housing_instability`, `sdoh_food_insecurity`, `sdoh_substance_use`, `sdoh_limited_social_support`, `sdoh_unemployment`, `sdoh_transportation_barrier` — six binary flags, each 1 if the corresponding social determinant of health was mentioned in the note and 0 otherwise

**Dropped (3 columns):**
- `hadm_id` — the join key used to match patients across datasets; not a clinical signal
- `sdoh_evidence` — a free-text field containing the raw sentence excerpts the LLM used to justify its SDOH flags; not a structured input that a tree model can use
- `discharge_risk_indicators_text` — similarly, a free-text list of the risk phrases extracted from the note; dropped for the same reason

The two text columns capture potentially useful clinical language but cannot be passed directly to XGBoost. They are kept in the dataframe for reference but excluded from the feature set.

## Feature preparation

Same feature set as the main classifier notebook, with the same encoding approach. For Model B we add the 9 discharge note features on top.


In [ ]:
cols_to_drop = [
    "subject_id", "hadm_id", "admittime", "dischtime",
    "readmitted_30d", "race_x_sex", "race_x_insurance",
    "race_clean", "insurance_clean", "sex", "admission_type",
    "hospital_expire_flag", "split",
    "psychiatric_complexity", "sdoh_evidence", "discharge_risk_indicators_text",
]

cohort["sex_enc"]            = cohort["sex"].map({"Male": 0, "Female": 1}).fillna(-1)
cohort["race_enc"]           = pd.Categorical(cohort["race_clean"]).codes
cohort["insurance_enc"]      = pd.Categorical(cohort["insurance_clean"]).codes
cohort["admission_type_enc"] = pd.Categorical(cohort["admission_type"]).codes

psych_map = {"low": 0, "medium": 1, "high": 2, "unknown": -1}
cohort["psychiatric_complexity_enc"] = cohort["psychiatric_complexity"].map(psych_map).fillna(-1)

llm_feature_cols = [
    "comorbidity_burden_score",
    "psychiatric_complexity_enc",
    "discharge_risk_indicator_count",
    "sdoh_housing_instability",
    "sdoh_food_insecurity",
    "sdoh_substance_use",
    "sdoh_limited_social_support",
    "sdoh_unemployment",
    "sdoh_transportation_barrier",
]

tabular_feature_cols = [c for c in cohort.columns if c not in cols_to_drop
                        and c not in llm_feature_cols]
tabular_feature_cols = list(dict.fromkeys(tabular_feature_cols))

print("tabular features:", len(tabular_feature_cols))
print("llm features:", len(llm_feature_cols))
print("total for model B:", len(tabular_feature_cols) + len(llm_feature_cols))

## Train / val / test split

Splitting 70/15/15. The val set is used for early stopping during XGBoost training and for setting the enrollment threshold. The test set is held out.

In [ ]:
train_idx, temp_idx = train_test_split(
    cohort.index,
    test_size=0.30,
    stratify=cohort["readmitted_30d"],
    random_state=random_seed
)

val_idx, test_idx = train_test_split(
    temp_idx,
    test_size=0.50,
    stratify=cohort.loc[temp_idx, "readmitted_30d"],
    random_state=random_seed
)

train_cohort = cohort.loc[train_idx].reset_index(drop=True)
val_cohort   = cohort.loc[val_idx].reset_index(drop=True)
test_cohort  = cohort.loc[test_idx].reset_index(drop=True)

target = "readmitted_30d"

y_train = train_cohort[target]
y_val   = val_cohort[target]
y_test  = test_cohort[target]

print("train:", len(train_cohort), "| readmission rate:", round(y_train.mean() * 100, 1), "%")
print("val:  ", len(val_cohort),   "| readmission rate:", round(y_val.mean() * 100, 1), "%")
print("test: ", len(test_cohort),  "| readmission rate:", round(y_test.mean() * 100, 1), "%")

## Fairness metric helper

Same function used throughout the project.

In [ ]:
def compute_fairness_metrics(df, group_col, enrolled_col, outcome_col):
    results = []
    for group in df[group_col].unique():
        subset      = df[df[group_col] == group]
        readmitted  = subset[subset[outcome_col] == 1]
        not_readmit = subset[subset[outcome_col] == 0]
        results.append({
            "group":           group,
            "n":               len(subset),
            "enrollment_rate": round(subset[enrolled_col].mean(), 3),
            "tpr":             round(readmitted[enrolled_col].mean(), 3) if len(readmitted) > 0 else 0,
            "fpr":             round(not_readmit[enrolled_col].mean(), 3) if len(not_readmit) > 0 else 0,
        })
    out = pd.DataFrame(results).sort_values("enrollment_rate", ascending=False)
    rates = out["enrollment_rate"]
    dp_ratio = round(rates.min() / rates.max(), 3) if rates.max() > 0 else 0
    return out, dp_ratio

## Model A: tabular features only

XGBoost trained from scratch on the 2,790-patient training set using the same 35 tabular features as the main classifier. Optuna tunes the hyperparameters against the val set.

In [ ]:
x_train_a = train_cohort[tabular_feature_cols]
x_val_a   = val_cohort[tabular_feature_cols]
x_test_a  = test_cohort[tabular_feature_cols]

def objective_a(trial):
    params = {
        "n_estimators":          trial.suggest_int("n_estimators", 100, 500),
        "learning_rate":         trial.suggest_float("learning_rate", 0.01, 0.1),
        "max_depth":             trial.suggest_int("max_depth", 3, 8),
        "subsample":             trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree":      trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "min_child_weight":      trial.suggest_int("min_child_weight", 1, 10),
        "scale_pos_weight":      (y_train == 0).sum() / (y_train == 1).sum(),
        "random_state":          random_seed,
        "eval_metric":           "auc",
        "early_stopping_rounds": 20,
    }
    m = xgb.XGBClassifier(**params, verbosity=0)
    m.fit(x_train_a, y_train, eval_set=[(x_val_a, y_val)], verbose=False)
    return roc_auc_score(y_val, m.predict_proba(x_val_a)[:, 1])

study_a = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=random_seed)
)
study_a.optimize(objective_a, n_trials=30, show_progress_bar=True)

print("best AUROC (val):", round(study_a.best_value, 4))
print("best params:", study_a.best_params)

In [ ]:
model_a = xgb.XGBClassifier(
    **study_a.best_params,
    scale_pos_weight=(y_train == 0).sum() / (y_train == 1).sum(),
    random_state=random_seed,
    eval_metric="auc",
    early_stopping_rounds=20,
    verbosity=0
)
model_a.fit(x_train_a, y_train, eval_set=[(x_val_a, y_val)], verbose=False)

val_preds_a  = model_a.predict_proba(x_val_a)[:, 1]
test_preds_a = model_a.predict_proba(x_test_a)[:, 1]

iso_a = IsotonicRegression(out_of_bounds="clip")
iso_a.fit(val_preds_a, y_val)
val_cal_a  = iso_a.predict(val_preds_a)
test_cal_a = iso_a.predict(test_preds_a)

threshold_a = pd.Series(val_cal_a).quantile(0.90)

val_enrolled_a = (val_cal_a >= threshold_a).mean()
test_cohort["enrolled_a"] = (test_cal_a >= threshold_a).astype(int)

metrics_a, dp_ratio_a = compute_fairness_metrics(
    test_cohort, "race_clean", "enrolled_a", "readmitted_30d"
)

auroc_a = roc_auc_score(y_test, test_cal_a)

print("Model A (tabular only — trained from scratch)")
print("AUROC:   ", round(auroc_a, 4))
print("dp_ratio:", dp_ratio_a)
print("enrollment rate (val / threshold target):", round(val_enrolled_a * 100, 1), "%")
print("enrollment rate (test / reported):        ", round(test_cohort["enrolled_a"].mean() * 100, 1), "%")
print("\nenrollment by race:")
print(metrics_a.to_string(index=False))

## Model B: tabular + discharge note features

Same setup as Model A but with the 9 discharge note features added to the feature set. Optuna tunes separately so both models get the best possible hyperparameters for their respective feature sets.

In [ ]:
all_feature_cols = tabular_feature_cols + llm_feature_cols

x_train_b = train_cohort[all_feature_cols].fillna(0)
x_val_b   = val_cohort[all_feature_cols].fillna(0)
x_test_b  = test_cohort[all_feature_cols].fillna(0)

def objective_b(trial):
    params = {
        "n_estimators":          trial.suggest_int("n_estimators", 100, 500),
        "learning_rate":         trial.suggest_float("learning_rate", 0.01, 0.1),
        "max_depth":             trial.suggest_int("max_depth", 3, 8),
        "subsample":             trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree":      trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "min_child_weight":      trial.suggest_int("min_child_weight", 1, 10),
        "scale_pos_weight":      (y_train == 0).sum() / (y_train == 1).sum(),
        "random_state":          random_seed,
        "eval_metric":           "auc",
        "early_stopping_rounds": 20,
    }
    m = xgb.XGBClassifier(**params, verbosity=0)
    m.fit(x_train_b, y_train, eval_set=[(x_val_b, y_val)], verbose=False)
    return roc_auc_score(y_val, m.predict_proba(x_val_b)[:, 1])

study_b = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=random_seed)
)
study_b.optimize(objective_b, n_trials=30, show_progress_bar=True)

print("best AUROC (val):", round(study_b.best_value, 4))
print("best params:", study_b.best_params)

In [ ]:
model_b = xgb.XGBClassifier(
    **study_b.best_params,
    scale_pos_weight=(y_train == 0).sum() / (y_train == 1).sum(),
    random_state=random_seed,
    eval_metric="auc",
    early_stopping_rounds=20,
    verbosity=0
)
model_b.fit(x_train_b, y_train, eval_set=[(x_val_b, y_val)], verbose=False)

val_preds_b  = model_b.predict_proba(x_val_b)[:, 1]
test_preds_b = model_b.predict_proba(x_test_b)[:, 1]

iso_b = IsotonicRegression(out_of_bounds="clip")
iso_b.fit(val_preds_b, y_val)
val_cal_b  = iso_b.predict(val_preds_b)
test_cal_b = iso_b.predict(test_preds_b)

threshold_b = pd.Series(val_cal_b).quantile(0.90)

val_enrolled_b = (val_cal_b >= threshold_b).mean()
test_cohort["enrolled_b"] = (test_cal_b >= threshold_b).astype(int)

metrics_b, dp_ratio_b = compute_fairness_metrics(
    test_cohort, "race_clean", "enrolled_b", "readmitted_30d"
)

auroc_b = roc_auc_score(y_test, test_cal_b)

print("Model B (tabular + discharge note features — trained from scratch)")
print("AUROC:   ", round(auroc_b, 4))
print("dp_ratio:", dp_ratio_b)
print("enrollment rate (val / threshold target):", round(val_enrolled_b * 100, 1), "%")
print("enrollment rate (test / reported):        ", round(test_cohort["enrolled_b"].mean() * 100, 1), "%")
print("\nenrollment by race:")
print(metrics_b.to_string(index=False))

## Feature importance for Model B

Checking where the discharge note features rank relative to the tabular features. This tells us whether XGBoost actually used them and which ones drove the most signal.

In [ ]:
importance_b = pd.DataFrame({
    "feature":    all_feature_cols,
    "importance": model_b.feature_importances_
}).sort_values("importance", ascending=False)

print("top 20 features:")
print(importance_b.head(20).to_string(index=False))

print("\ndischarge note feature importances:")
print(importance_b[importance_b["feature"].isin(llm_feature_cols)].to_string(index=False))

## Comparison

Side by side summary of both models trained from scratch on the same 2,790 patients.

In [ ]:
print("=" * 50)
print("SUMMARY: Stage 1 Scratch Training Sub-Analysis")
print(f"Cohort: {len(test_cohort)} patients (held-out test set)")
print("=" * 50)
print(f"{'Metric':<25} {'Model A':>12} {'Model B':>12} {'Change':>12}")
print("-" * 50)
print(f"{'AUROC':<25} {auroc_a:>12.4f} {auroc_b:>12.4f} {auroc_b - auroc_a:>+12.4f}")
print(f"{'dp_ratio':<25} {dp_ratio_a:>12.3f} {dp_ratio_b:>12.3f} {dp_ratio_b - dp_ratio_a:>+12.3f}")
print("=" * 50)

print("\nenrollment rates by race:")
comparison = metrics_a[["group", "n", "enrollment_rate"]].rename(
    columns={"enrollment_rate": "enroll_rate_a"}
).merge(
    metrics_b[["group", "enrollment_rate"]].rename(
        columns={"enrollment_rate": "enroll_rate_b"}
    ),
    on="group"
)
comparison["change"] = (comparison["enroll_rate_b"] - comparison["enroll_rate_a"]).round(3)
print(comparison.to_string(index=False))



## Saving results

In [ ]:
test_cohort[["hadm_id", "race_clean", "readmitted_30d",
             "enrolled_a", "enrolled_b"]].to_parquet(
    "/kaggle/working/stage1_scratch_results.parquet", index=False
)

comparison.to_parquet("/kaggle/working/stage1_scratch_comparison.parquet", index=False)

print("saved stage1_scratch_results.parquet")
print("saved stage1_scratch_comparison.parquet")